# Simulações e figuras do artigo — intrusão salina 1D no Rio São Mateus

Este notebook reproduz, de forma didática, as simulações e as **sete figuras** usadas no artigo. Ele utiliza diretamente o módulo `salt_intrusion_1d`; o método numérico não é reimplementado nas células.

## Objetivos

1. executar os cenários de controle e crítico, com $Q=10$ e $Q=2\,\mathrm{m^3\,s^{-1}}$;
2. examinar a dinâmica após 3 e 60 ciclos de maré;
3. calcular $\overline{L}_s$ e $L_s^{\max}$ no último ciclo dos horizontes selecionados;
4. comparar dispersão constante e dependente da velocidade;
5. salvar as figuras com os mesmos nomes utilizados no manuscrito.

> **Escopo.** As forçantes são periódicas e idealizadas. Os resultados constituem uma aplicação didática e uma verificação computacional do modelo; não correspondem a uma calibração ou validação quantitativa do Rio São Mateus.

## 1. Instalação e organização dos arquivos

### Execução local

Mantenha este notebook na pasta `notebooks/` do pacote e execute a célula abaixo.

### Google Colab

Envie `salt_intrusion_1d_v1.0.1.zip` para `/content`. A célula detectará o Colab, descompactará o pacote e instalará o módulo automaticamente.

As figuras serão gravadas na pasta `results_article_figures/`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    archive = Path("/content/salt_intrusion_1d_v1.0.1.zip")
    if not archive.exists():
        raise FileNotFoundError(
            "Envie salt_intrusion_1d_v1.0.1.zip para /content e execute novamente."
        )
    project_dir = Path("/content/salt_intrusion_1d_v1.0.1")
    project_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as compressed:
        compressed.extractall(project_dir)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(project_dir)]
    )
else:
    project_dir = Path("..").resolve()
    source_dir = project_dir / "src"
    if not source_dir.exists():
        raise FileNotFoundError(
            "A pasta src/ não foi encontrada. Mantenha o notebook em notebooks/."
        )
    sys.path.insert(0, str(source_dir))

output_dir = project_dir / "results_article_figures"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Pacote: {project_dir}")
print(f"Figuras: {output_dir}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:  # permite validar também como script Python comum
    display = print

from salt_intrusion_1d import simulate
from salt_intrusion_1d.article_update import article_config
from salt_intrusion_1d.domain_sensitivity import CAPTURE_FROM_MOUTH_KM
from salt_intrusion_1d.experiment import MOUTH_OFFSET_KM

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})

DISCHARGES = (10.0, 2.0)
COLORS = {10.0: "tab:blue", 2.0: "tab:orange"}
STYLES = {10.0: "-", 2.0: "--"}
HORIZONS = (3, 10, 20, 30, 40, 50, 60)
TIDAL_PERIOD_H = 12.4
AREA_M2 = 1050.0

## 2. Modelo e discretização

No domínio computacional $x\in[0,L]$, resolve-se

$$
\frac{\partial C}{\partial t}+v(t)\frac{\partial C}{\partial x}
=D(t)\frac{\partial^2 C}{\partial x^2},
$$

com

$$
v(t)=-\frac{Q}{A}+U_m\sin(\omega t),
\qquad
D(t)=D_0+\kappa |v(t)|B.
$$

Adotam-se Euler implícito no tempo, diferenças *upwind* para a advecção e diferenças centrais para a dispersão. Em $x=0$, a salinidade marítima é prescrita. Em $x=L$, usa-se Neumann homogênea quando $v\geq0$ e Robin–Danckwerts quando $v<0$.

A entrada numérica $x=0$ representa uma seção situada a $10\,\mathrm{km}$ da foz. Portanto, as figuras usam

$$
x_{\mathrm{foz}}=x+10\,\mathrm{km}.
$$

O comprimento de intrusão é definido pela posição mais a montante em que $C\geq C_{\mathrm{lim}}$, com $C_{\mathrm{lim}}=0{,}5\,\mathrm{PSU}$, usando interpolação linear na travessia do limiar.

## 3. Configuração final do artigo

Os cálculos usam $L=50\,\mathrm{km}$, $A=1050\,\mathrm{m^2}$, $D_0=30\,\mathrm{m^2/s}$, $\kappa=0{,}25$, $\Delta x=12{,}5\,\mathrm{m}$ e $\Delta t=7{,}5\,\mathrm{s}$. A condição inicial é água doce no interior do domínio, $C(x,0)=0$.

In [ ]:
configs = {
    Q: article_config(
        Q,
        length_km=50.0,
        cycles=60,
        dx_m=12.5,
        dt_s=7.5,
        base_dispersion_m2_s=30.0,
        dispersion_kappa=0.25,
        dispersion_mode="velocity_dependent",
        store_every_steps=60,  # perfil a cada 7,5 min e em todo fim de ciclo
    )
    for Q in DISCHARGES
}

config_table = pd.DataFrame([
    {
        "Q (m³/s)": Q,
        "L (km)": cfg.length_m / 1000,
        "A (m²)": cfg.cross_section_area_m2,
        "Δx (m)": cfg.dx_m,
        "Δt (s)": cfg.dt_s,
        "D₀ (m²/s)": cfg.base_dispersion_m2_s,
        "κ": cfg.dispersion_kappa,
        "células": cfg.n_cells,
        "passos": cfg.n_steps,
    }
    for Q, cfg in configs.items()
])
display(config_table)

## 4. Execução dos dois cenários

Cada simulação avança 60 ciclos completos. Um único resultado contém toda a série temporal de $L_s(t)$ e perfis armazenados nos finais dos ciclos; por isso, não é necessário repetir a simulação para cada horizonte da tabela.

> A célula seguinte é a mais demorada do notebook.

In [ ]:
results = {}
for Q in DISCHARGES:
    print(f"Executando Q={Q:g} m³/s ...")
    results[Q] = simulate(configs[Q])
    print(
        f"  concluído: L̄s(60)="
        f"{results[Q].mean_intrusion_last_cycle_m()/1000 + MOUTH_OFFSET_KM:.2f} km; "
        f"Ls_max(60)="
        f"{results[Q].max_intrusion_last_cycle_m()/1000 + MOUTH_OFFSET_KM:.2f} km"
    )

## 5. Funções auxiliares para pós-processamento

As funções abaixo apenas selecionam perfis, calculam métricas e padronizam a apresentação. Elas não alteram a solução numérica.

In [ ]:
def profile_at_cycle(result, cycle):
    target = cycle * result.config.tidal_period_s
    index = np.flatnonzero(np.isclose(result.stored_times_s, target, atol=1e-8, rtol=0.0))
    if index.size != 1:
        raise RuntimeError(f"Perfil do ciclo {cycle} não foi armazenado de forma única.")
    return result.stored_profiles_psu[int(index[0])]


def cycle_metrics(result, cycle):
    period = result.config.tidal_period_s
    end = cycle * period
    start = (cycle - 1) * period
    mask = (result.times_s >= start) & (result.times_s <= end)
    time = result.times_s[mask]
    length = result.intrusion_length_m[mask]
    mean_from_mouth = np.trapezoid(length, time) / period / 1000 + MOUTH_OFFSET_KM
    max_from_mouth = np.max(length) / 1000 + MOUTH_OFFSET_KM
    return mean_from_mouth, max_from_mouth


def save_and_show(fig, filename):
    path = output_dir / filename
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Salva em: {path}")
    return path


def add_threshold_and_capture(ax, *, threshold=False, capture=False):
    if threshold:
        ax.axhline(0.5, color="0.35", linestyle=":", linewidth=1.2,
                   label=r"$C_{\mathrm{lim}}=0{,}5\,\mathrm{PSU}$")
    if capture:
        ax.axhline(CAPTURE_FROM_MOUTH_KM, color="tab:red", linestyle=":",
                   linewidth=1.3, label="Captação (41 km)")

## 6. Dinâmica inicial: três ciclos

O perfil no final do terceiro ciclo mostra a propagação inicial da salinidade a partir da extremidade marítima. A série de $L_s(t)$ evidencia simultaneamente o avanço transiente e a oscilação mareal.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for Q in DISCHARGES:
    result = results[Q]
    ax.plot(
        result.x_m / 1000 + MOUTH_OFFSET_KM,
        profile_at_cycle(result, 3),
        STYLES[Q], color=COLORS[Q], linewidth=2.2,
        label=rf"$Q={Q:g}\,\mathrm{{m^3/s}}$",
    )
ax.axhline(0.5, color="0.35", linestyle=":", linewidth=1.2,
           label=r"$C_{\mathrm{lim}}$")
ax.set(xlabel="Distância da foz (km)", ylabel="Salinidade (PSU)",
       title="Perfis de salinidade ao final do terceiro ciclo", xlim=(10, 35))
ax.legend()
save_and_show(fig, "fig-RSM-3-ciclos-sal.png")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
end = 3 * configs[10.0].tidal_period_s
for Q in DISCHARGES:
    result = results[Q]
    mask = result.times_s <= end
    ax.plot(
        result.times_s[mask] / 3600,
        result.intrusion_length_m[mask] / 1000 + MOUTH_OFFSET_KM,
        STYLES[Q], color=COLORS[Q], linewidth=1.8,
        label=rf"$Q={Q:g}\,\mathrm{{m^3/s}}$",
    )
ax.set(xlabel="Tempo (h)", ylabel="Distância da frente à foz (km)",
       title="Comprimento de intrusão nos três primeiros ciclos")
ax.legend()
save_and_show(fig, "fig-RSM-3-ciclos-Ls.png")

In [ ]:
table_3 = pd.DataFrame([
    {
        "Q (m³/s)": Q,
        "L̄s no 3º ciclo (km da foz)": cycle_metrics(results[Q], 3)[0],
        "Ls_max no 3º ciclo (km da foz)": cycle_metrics(results[Q], 3)[1],
    }
    for Q in DISCHARGES
]).round(2)
display(table_3)

## 7. Estrutura espacial e evolução até o 60º ciclo

As figuras seguintes correspondem ao horizonte de $744\,\mathrm{h}$, aproximadamente 31 dias. O perfil é mostrado no instante final; $L_s(t)$ é mostrado ao longo de toda a simulação.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for Q in DISCHARGES:
    result = results[Q]
    ax.plot(
        result.x_m / 1000 + MOUTH_OFFSET_KM,
        result.final_profile_psu,
        STYLES[Q], color=COLORS[Q], linewidth=2.2,
        label=rf"$Q={Q:g}\,\mathrm{{m^3/s}}$",
    )
ax.axhline(0.5, color="0.35", linestyle=":", linewidth=1.2,
           label=r"$C_{\mathrm{lim}}$")
ax.axvline(CAPTURE_FROM_MOUTH_KM, color="tab:red", linestyle=":",
           linewidth=1.3, label="Captação (41 km)")
ax.set(xlabel="Distância da foz (km)", ylabel="Salinidade (PSU)",
       title="Perfis de salinidade ao final de 60 ciclos", xlim=(10, 60))
ax.legend()
save_and_show(fig, "fig-RSM-60-ciclos-sal.png")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for Q in DISCHARGES:
    result = results[Q]
    ax.plot(
        result.times_s / (24 * 3600),
        result.intrusion_length_m / 1000 + MOUTH_OFFSET_KM,
        STYLES[Q], color=COLORS[Q], linewidth=1.25,
        label=rf"$Q={Q:g}\,\mathrm{{m^3/s}}$",
    )
ax.axhline(CAPTURE_FROM_MOUTH_KM, color="tab:red", linestyle=":",
           linewidth=1.3, label="Captação (41 km)")
ax.set(xlabel="Tempo (dias)", ylabel="Distância da frente à foz (km)",
       title="Comprimento de intrusão ao longo de 60 ciclos")
ax.legend()
save_and_show(fig, "fig-RSM-60-ciclos-Ls.png")

## 8. Métricas nos horizontes selecionados

Para cada horizonte, as métricas são calculadas exclusivamente no último ciclo completo:

$$
\overline{L}_s=\frac{1}{T}\int_{t_f-T}^{t_f}L_s(t)\,dt,
\qquad
L_s^{\max}=\max_{t\in[t_f-T,t_f]}L_s(t).
$$

In [ ]:
rows = []
for Q in DISCHARGES:
    for cycle in HORIZONS:
        mean_length, max_length = cycle_metrics(results[Q], cycle)
        rows.append({
            "Q (m³/s)": Q,
            "ciclos": cycle,
            "tempo (dias)": cycle * TIDAL_PERIOD_H / 24,
            "L̄s (km da foz)": mean_length,
            "Ls_max (km da foz)": max_length,
        })

landmarks = pd.DataFrame(rows)
display(landmarks.round(2))
landmarks.to_csv(output_dir / "metricas_horizontes_artigo.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for Q in DISCHARGES:
    subset = landmarks[landmarks["Q (m³/s)"] == Q]
    ax.plot(
        subset["ciclos"], subset["L̄s (km da foz)"],
        STYLES[Q], color=COLORS[Q], marker="o" if Q == 10 else "s",
        linewidth=2.2, label=rf"$Q={Q:g}\,\mathrm{{m^3/s}}$",
    )
ax.axhline(CAPTURE_FROM_MOUTH_KM, color="tab:red", linestyle=":",
           linewidth=1.4, label="Captação (41 km)")
ax.set(xlabel="Número de ciclos de maré", ylabel=r"$\overline{L}_s$ (km da foz)",
       title="Evolução do comprimento médio de intrusão")
ax.legend()
save_and_show(fig, "Evolucao_intrusao_ciclos.png")

## 9. Influência do modelo de dispersão

Mantendo $Q=10\,\mathrm{m^3/s}$ e $D_0=30\,\mathrm{m^2/s}$, compara-se:

$$
D(t)=D_0
\qquad\text{e}\qquad
D(t)=D_0+\kappa |v(t)|B,
\quad \kappa=0{,}25.
$$

O resultado dependente da velocidade já foi calculado. Resta executar apenas o caso com dispersão constante.

In [ ]:
constant_config = article_config(
    10.0,
    length_km=50.0,
    cycles=60,
    dx_m=12.5,
    dt_s=7.5,
    base_dispersion_m2_s=30.0,
    dispersion_kappa=0.0,
    dispersion_mode="constant",
    store_every_steps=60,
)
print("Executando Q=10 m³/s com dispersão constante ...")
constant_result = simulate(constant_config)
dispersion_results = {
    "constant": constant_result,
    "velocity_dependent": results[10.0],
}
for name, result in dispersion_results.items():
    print(
        f"{name:>18}: L̄s="
        f"{result.mean_intrusion_last_cycle_m()/1000 + MOUTH_OFFSET_KM:.2f} km; "
        f"Ls_max={result.max_intrusion_last_cycle_m()/1000 + MOUTH_OFFSET_KM:.2f} km"
    )

In [ ]:
dispersion_labels = {
    "constant": r"$D(t)=D_0$",
    "velocity_dependent": r"$D(t)=D_0+\kappa|v(t)|B$",
}
dispersion_styles = {"constant": "--", "velocity_dependent": "-"}

fig, ax = plt.subplots(figsize=(7.2, 4.4))
for mode in ("constant", "velocity_dependent"):
    result = dispersion_results[mode]
    ax.plot(
        result.x_m / 1000 + MOUTH_OFFSET_KM,
        result.final_profile_psu,
        dispersion_styles[mode], linewidth=2.2, label=dispersion_labels[mode],
    )
ax.axhline(0.5, color="0.35", linestyle=":", linewidth=1.2,
           label=r"$C_{\mathrm{lim}}$")
ax.set(xlabel="Distância da foz (km)", ylabel="Salinidade (PSU)",
       title=r"Perfil final para $Q=10\,\mathrm{m^3/s}$", xlim=(10, 60))
ax.legend()
save_and_show(fig, "fig-RSM-60-ciclos-sal-D.png")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for mode in ("constant", "velocity_dependent"):
    result = dispersion_results[mode]
    ax.plot(
        result.times_s / (24 * 3600),
        result.intrusion_length_m / 1000 + MOUTH_OFFSET_KM,
        dispersion_styles[mode], linewidth=1.25, label=dispersion_labels[mode],
    )
ax.set(xlabel="Tempo (dias)", ylabel="Distância da frente à foz (km)",
       title=r"Efeito da dispersão para $Q=10\,\mathrm{m^3/s}$")
ax.legend()
save_and_show(fig, "fig-RSM-60-ciclos-Ls-D.png")

## 10. Conferência dos arquivos produzidos

A célula final verifica se as sete figuras do manuscrito e a tabela de métricas foram efetivamente gravadas.

In [ ]:
expected = [
    "fig-RSM-3-ciclos-sal.png",
    "fig-RSM-3-ciclos-Ls.png",
    "fig-RSM-60-ciclos-sal.png",
    "fig-RSM-60-ciclos-Ls.png",
    "Evolucao_intrusao_ciclos.png",
    "fig-RSM-60-ciclos-sal-D.png",
    "fig-RSM-60-ciclos-Ls-D.png",
    "metricas_horizontes_artigo.csv",
]
manifest = pd.DataFrame([
    {
        "arquivo": name,
        "gerado": (output_dir / name).exists(),
        "tamanho (kB)": round((output_dir / name).stat().st_size / 1024, 1)
        if (output_dir / name).exists() else np.nan,
    }
    for name in expected
])
display(manifest)
assert manifest["gerado"].all(), "Há arquivos esperados que não foram gerados."
print("Verificação concluída: todos os arquivos foram produzidos.")